# Web Agent Action Prediction — Colab Pipeline

**최신화: 2026-05-08**

**전제**: Google Drive의 `DRIVE_ROOT` 아래에 준비:
- `my_code.zip` — `src/` 4개 파일을 zip 압축한 것
- `data/train.csv`, `data/test.csv`, `data/somenna_submission.csv`

**순서**: GPU 확인 → 설치 → 코드/데이터 → GPU 패치 → 학습 → 백업 → 추론 → 점검 → 저장

## 1. GPU 확인

In [ ]:
import subprocess
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader']).decode().strip()
print('GPU:', gpu)
assert any(g in gpu for g in ['T4', 'A100', 'L4', 'V100']), f'지원하지 않는 GPU: {gpu}'
!nvidia-smi

GPU: NVIDIA A100-SXM4-80GB
Thu May  7 22:18:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+--------------------

## 2. 패키지 설치
런타임 재시작 불필요.

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --no-deps "trl>=0.21" peft accelerate bitsandbytes
!pip install pandas tqdm scikit-learn

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-51mphe_2/unsloth_85da6cf7b12f481aaf47c171db53ed03
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-51mphe_2/unsloth_85da6cf7b12f481aaf47c171db53ed03
  Resolved https://github.com/unslothai/unsloth.git to commit d1f9ab659fc5d3309e9e40166be309369bedf852
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.5.2-py3-none-any.whl size=31540367 sha256=ddbb994f522b67aae5900e4f1590833f0624d09064ebb01867d75c479dcead8c
  Stored in directory: /tmp/pip-ephem-wheel-cache-ezc3csru/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth


## 3. Drive 마운트 + 코드/데이터 배치

`DRIVE_ROOT` 아래에 다음 파일이 있어야 합니다:
- `my_code.zip` — `src/` 폴더 안의 4개 파일(`preprocess.py`, `retrieval.py`, `train.py`, `inference.py`)을 zip 압축
- `data/train.csv`, `data/test.csv`, `data/somenna_submission.csv`

In [ ]:
from google.colab import files
import os, zipfile

os.makedirs('/content/src', exist_ok=True)
os.makedirs('/content/data', exist_ok=True)

uploaded = files.upload()  # my_code.zip + 3개 csv 선택

# zip 압축 해제
with zipfile.ZipFile('my_code.zip') as z:
    z.extractall('/content/src/')

# csv 이동
import shutil
for f in ['train.csv', 'test.csv', 'somenna_submission.csv']:
    if os.path.exists(f):
        shutil.move(f, f'/content/data/{f}')

!ls /content/src && echo '---' && ls /content/data

Saving my_code.zip to my_code.zip
inference.py  preprocess.py  retrieval.py  train.py
---


## 4. GPU별 설정 자동 패치

| GPU | VRAM | batch | grad_accum | 유효배치 | max_steps | lr | 학습시간 | inf batch |
|-----|------|-------|------------|---------|-----------|-----|---------|----------|
| **A100** | 40GB | 32 | 1 | 32 | **3000** | 4e-4 | ~3.5h | 32 |
| L4  | 24GB | 2 | 8 | 16 | **1500** | 2e-4 | ~5h | 8 |
| T4  | 16GB | 1 | 16 | 16 | **1000** | 2e-4 | ~5h | 4 |

**변경사항 (2026-05-08)**:
- CoT(Chain-of-Thought) 적용 — task-aware 추론 문장 + JSON, max_new_tokens 128
- `max_steps` 증가 (A100 기준 2400→3000)
- `warmup_ratio` 0.1→0.05, `save_steps` 500→300
- `SHUFFLE_AUGMENT_N` 2→3 (real_web 7배, workflow 4배 augmentation)
- `extract_value_from_task`: 날짜 regex를 date 필드 확인된 경우에만 반환
- `enforce_consistency`: fallback 후 target_id 안전 대체, SELECT value 첫 옵션 폴백 추가

> **A100 권장**: 학습+추론 한 세션 안에 완결 가능

In [ ]:
import subprocess, re

gpu_name = subprocess.check_output(
    ["nvidia-smi","--query-gpu=name","--format=csv,noheader"]
).decode().strip()

is_a100 = "A100" in gpu_name
is_l4   = "L4"   in gpu_name
profile = "A100" if is_a100 else ("L4" if is_l4 else "T4")
print(f"GPU: {gpu_name} -> {profile} 프로파일 적용")

tp = "/content/src/train.py"
ip = "/content/src/inference.py"

if is_a100:
    s = open(tp).read()
    s = re.sub(r"per_device_train_batch_size\s*=\s*\d+", "per_device_train_batch_size = 32", s)
    s = re.sub(r"gradient_accumulation_steps\s*=\s*\d+", "gradient_accumulation_steps = 1", s)
    s = re.sub(r"learning_rate\s*=\s*[\d.e+-]+",         "learning_rate = 4e-4", s)
    s = re.sub(r"max_steps\s*=\s*\d+",                   "max_steps = 3000", s)
    s = re.sub(r"warmup_ratio\s*=\s*[\d.]+",             "warmup_ratio = 0.05", s)
    s = re.sub(r"save_steps\s*=\s*\d+",                  "save_steps = 300", s)
    s = re.sub(r"dataset_num_proc\s*=\s*\d+",            "dataset_num_proc = 4", s)
    open(tp, "w").write(s)
    s = open(ip).read()
    s = re.sub(r"^BATCH_SIZE\s*=\s*\d+", "BATCH_SIZE = 32", s, flags=re.M)
    open(ip, "w").write(s)
    print("A100 패치 완료: train(batch=32, grad=1, lr=4e-4, steps=3000, warmup=0.05) / inference(batch=32)")

elif is_l4:
    s = open(tp).read()
    s = re.sub(r"per_device_train_batch_size\s*=\s*\d+", "per_device_train_batch_size = 2", s)
    s = re.sub(r"gradient_accumulation_steps\s*=\s*\d+", "gradient_accumulation_steps = 8", s)
    s = re.sub(r"learning_rate\s*=\s*[\d.e+-]+",         "learning_rate = 2e-4", s)
    s = re.sub(r"max_steps\s*=\s*\d+",                   "max_steps = 1500", s)
    s = re.sub(r"warmup_ratio\s*=\s*[\d.]+",             "warmup_ratio = 0.05", s)
    s = re.sub(r"save_steps\s*=\s*\d+",                  "save_steps = 150", s)
    s = re.sub(r"dataset_num_proc\s*=\s*\d+",            "dataset_num_proc = 4", s)
    open(tp, "w").write(s)
    s = open(ip).read()
    s = re.sub(r"^BATCH_SIZE\s*=\s*\d+", "BATCH_SIZE = 8", s, flags=re.M)
    open(ip, "w").write(s)
    print("L4 패치 완료: train(batch=2, grad=8, lr=2e-4, steps=1500, warmup=0.05) / inference(batch=8)")

else:  # T4
    s = open(tp).read()
    s = re.sub(r"per_device_train_batch_size\s*=\s*\d+", "per_device_train_batch_size = 1", s)
    s = re.sub(r"gradient_accumulation_steps\s*=\s*\d+", "gradient_accumulation_steps = 16", s)
    s = re.sub(r"learning_rate\s*=\s*[\d.e+-]+",         "learning_rate = 2e-4", s)
    s = re.sub(r"max_steps\s*=\s*\d+",                   "max_steps = 1000", s)
    s = re.sub(r"warmup_ratio\s*=\s*[\d.]+",             "warmup_ratio = 0.05", s)
    s = re.sub(r"save_steps\s*=\s*\d+",                  "save_steps = 100", s)
    open(tp, "w").write(s)
    s = open(ip).read()
    s = re.sub(r"^BATCH_SIZE\s*=\s*\d+", "BATCH_SIZE = 4", s, flags=re.M)
    open(ip, "w").write(s)
    print("T4 패치 완료: train(batch=1, grad=16, lr=2e-4, steps=1000, warmup=0.05) / inference(batch=4)")

GPU: NVIDIA A100-SXM4-80GB -> A100 프로파일 적용
A100 패치 완료: train(batch=32, grad=1, lr=4e-4, steps=3000, warmup=0.05) / inference(batch=32)


## 5. 학습 (LoRA SFT + CoT)

- 모델: `Qwen3-8B` (4bit)
- LoRA: r=16, alpha=32, 7개 projection
- max_steps: **3000** (A100 기준), warmup_ratio=0.05, cosine scheduler
- SHUFFLE_AUGMENT_N: **3** (real_web 7배, workflow 4배 augmentation)
- 학습 제외: `site_2aa627db`
- **CoT**: 답변 = task-aware 추론 문장(최대 120자) + JSON `{"op", "choice", "value"}`
- **value 추출**: 날짜 regex는 date 타입 필드 확인된 경우에만 반환
- **Consistency Guard**: fallback 후 target_id 안전 대체, SELECT value 첫 옵션 폴백 추가

산출물: `/content/lora_model/`, `/content/outputs/eval_metrics.json`

In [ ]:
%cd /content
!python src/train.py

/content
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: NVIDIA A100-SXM4-80GB | VRAM: 79.3 GB
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
model.safetensors: 100% 6.07G/6.07G [00:12<00:00, 497MB/s]
Loading weights: 100% 399/399 [00:01<00:00, 215.79it/s, Materializing param=model.norm.weight]
generation_config.json: 100% 237/237 [00:00<00:00, 1.17MB/s]
config.json: 1.33kB [00:00, 3.49MB/s]
tokenizer_config.json: 10.5kB [00:00, 25.8MB/s]
vocab.json: 2.78MB [00:00, 83.3MB/s]
merges.txt: 

## 6. 학습 결과 확인

> **[Claude 공유 필수 #1]**  
> 아래 셀을 실행한 뒤 **출력 결과 전체를 복사해서 Claude에게 붙여넣기**  
> (workflow / real_web 각각의 op_acc / target_acc / value_acc / exact_match 값이 핵심)


In [2]:
import json
with open('/content/outputs/eval_metrics.json') as f:
    metrics = json.load(f)

print('=== 검증 결과 ===')
for key, m in metrics.items():
    if m.get('n', 0) == 0:
        continue
    print(f"\n[{key}] n={m['n']}")
    print(f"  op_acc      : {m['op_acc']:.3f}")
    print(f"  target_acc  : {m['target_id_acc']:.3f}")
    print(f"  value_acc   : {m['value_acc']:.3f}")
    print(f"  exact_match : {m['exact_match']:.3f}  ← 대회 기준")

FileNotFoundError: [Errno 2] No such file or directory: '/content/outputs/eval_metrics.json'

## 7. lora_model Drive 백업

In [ ]:
import os
!mkdir -p "$DRIVE_ROOT/lora_model"
!cp -r /content/lora_model/* "$DRIVE_ROOT/lora_model/"
!cp /content/outputs/eval_metrics.json "$DRIVE_ROOT/eval_metrics.json"
print('백업 완료')

백업 완료


## 8. 추론 → submission.csv

In [ ]:
%cd /content
!python src/inference.py

/content
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
1. Preparing retriever from train data...
2. Loading LLM...
==((====))==  Unsloth 2026.5.2: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 399/399 [00:01<00:00, 219.50it/s, Materializing param=model.norm.weight]
unsloth/Qwen3-8B-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.
3. Running streaming inference...
Chunks: 0it [00:00, ?it/s

## 9. 제출 파일 점검

> **[Claude 공유 필수 #2]**  
> 아래 셀을 실행한 뒤 **출력 결과 전체를 복사해서 Claude에게 붙여넣기**  
> (rows 수, op 분포, 빈 target_id, 결측값 확인)


In [1]:
import pandas as pd
df = pd.read_csv('/content/submission.csv')
print('rows:', len(df))
print('\nop 분포:')
print(df['op'].value_counts(dropna=False))
print('\n빈 target_id:', (df['target_id'].isna() | (df['target_id'] == '')).sum())
print('CLICK에 value 있음:', ((df['op'] == 'CLICK') & (df['value'].astype(str).str.strip() != '')).sum())
print('결측값:', df.isnull().sum().sum())
df.head(10)

FileNotFoundError: [Errno 2] No such file or directory: '/content/submission.csv'

## 10. submission.csv Drive 저장

In [ ]:
!cp /content/submission.csv "$DRIVE_ROOT/submission.csv"
print('Drive 저장 완료')

cp: cannot stat '/content/submission.csv': No such file or directory
Drive 저장 완료


In [ ]:
from google.colab import files
files.download('/content/submission.csv')

FileNotFoundError: Cannot find file: /content/submission.csv